# Data Summary + Analysis

## Set Up

Import libraries/packages + cleaned data

In [1]:
# Libraries/packages
import sys

sys.path.append("../")
from src.data_utils import get_feature_lists
from src.config import BASE_PATH
from src.summary_analysis import (
    generate_summary_table,
    get_analysis_df,
)
import warnings
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import pandas as pd

Data

In [3]:
## HOPKINS data
cleaned_data = pd.read_parquet(
    BASE_PATH / "data/raw/hopkins_ORN_extra_clean.parquet"
).drop("ID", axis=1)
X = cleaned_data.drop("ORN", axis=1)
y = cleaned_data["ORN"]

Re-order a bit

In [4]:
reordered_cols = [
    ##Pre-Op
    "AGE",
    "BMI",
    "SEX",
    "Diabetes",
    "ASA",
    "PRIOREX",
    "PRECT",
    ## Disease
    "RECUR",
    "SITE",
    "SIZE",
    "LYMPH",
    "STAGE",
    "DEFECT",
    "SECONDPRIMARY",
    ## Surg
    "LENGTH",
    "JEWER",
    "OSTEOTOMY",
    "PLATE",
    "FLAP",
    "TRANSFUS",
    "ISCHEMICTIME",
    "OPTIME",
    ## Immediate post-op
    "REOP",
    "LOHS",
    "POSTCT",
    "WOUNDINF",
    "HGB",
    "ALB",
    ## Long term post-op
    "EXPOSURE",
    "MEDUSED",
    "SURGUSED",
    "PLATETIME",
    "FOLLOWTIME",
    ## Misc
    "RADTIME",
]

X_ordered = X[reordered_cols].copy()

Classify features by data type

In [5]:
##Imported func from src
feature_lists = get_feature_lists(X_ordered)
binary_cols = feature_lists["Binary"]
numerical_cols = feature_lists["Numerical"]
nominal_cols = feature_lists["Nominal"]
ordinal_cols = feature_lists["Ordinal"]

Impute

In [ ]:
imputer = IterativeImputer(
    estimator=None,  # default = BayesianRidge
    initial_strategy="median",
    max_iter=10,
    sample_posterior=False,  # deterministic
)
X_imp = X_ordered.copy()
imputed_values = imputer.fit_transform(X_imp[numerical_cols])
X_imp[numerical_cols] = imputed_values
## Ensure no NAs
assert X_imp.isna().sum().sum() == 0

## Summary + Analysis

In [ ]:
data_type = "hopkins"
# Use fishers for all binary bc of low sample size
fish_dict = {data_type: binary_cols}
## Create separate tables for each dataset
final_tables_dict = {}  # Store tables by dataset name

print(f"Working on {data_type}...")

# Create all_categories specific to THIS dataset
all_categories = {}
for col in nominal_cols + binary_cols + ordinal_cols:
    all_categories[col] = X_imp[col].unique()
## Get summary
summary_df = generate_summary_table(
    X_df_final=X_imp,
    X_df_og=X_ordered,
    outcome_data=y,
    data_type=data_type,
    all_categories=all_categories,
    feature_dict=feature_lists,
)
# # Df containing univariable values (p-values, ORs w/ CIs)
analysis_df = get_analysis_df(
    df=X_imp,
    outcome_data=y,
    data_type=data_type,
    fish_dict=fish_dict,
)
# Verify indices match
try:
    assert set(analysis_df.index.to_list()) == set(summary_df.index.to_list())
except AssertionError:
    print(f"Mismatch in {data_type}:")
    print(
        "In analysis but not summary:",
        set(analysis_df.index.to_list()) - set(summary_df.index.to_list()),
    )
    print(
        "In summary but not analysis:",
        set(summary_df.index.to_list()) - set(analysis_df.index.to_list()),
    )
    raise AssertionError(f"Analysis and summary tables DO NOT match for {data_type}!")
# Join summary and analysis for this dataset
final_table = summary_df.join(analysis_df, how="left").fillna("NA")
print(f"Completed {data_type} table with shape {final_table.shape}")

Export

In [ ]:
## Set up path
export_path = BASE_PATH / "results" / "tables" / "summary_analysis.xlsx"
if export_path.exists():
    export_path.unlink()
    warnings.warn(f"Over-writing folder at path {export_path}")
export_path.parent.mkdir(exist_ok=True, parents=True)
## Export
final_table.to_excel(export_path)